In [ ]:
!pip install openai
!pip install python-dotenv

In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

In [ ]:
#load environment variables from .env file
load_dotenv()
token = os.getenv("HUGGINGFACE_TOKEN")

In [ ]:
client = OpenAI(
    base_url = "https://api-inference.huggingface.co/models/your-model-name",
    api_key = token
)

tool = [
    {
        "type": "function",
        "function": {
            "name": "find_object",
            "description": "Extract english keywords from the user's query and search the database for matching videos or images containing the object.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Free-text user query describing the object to find (e.g., 'find all red cars near the lake')"
                    }
                },
                "required": ["query"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get current temperature for a given location.",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "City and country e.g. Bogotá, Colombia",
                    }
                },
                "required": ["location"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_similarity",
            "description": "Search the database for multimedia items (images, videos, audio) most similar to the given query or reference input.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Text or reference content to find similar items. e.g. photos similar to this mountain sunrise."
                    },
                    "top_k": {
                        "type": "integer",
                        "description": "Number of top results to return"
                    }
                },
                "required": ["query", "top_k"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "generate_description",
            "description": "Generate a short, clear English description of the given multimedia content (image, video, or audio).",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Text or reference describing the content to be described. e.g. Describe this image of a busy street at night."
                    }
                },
                "required": ["query"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "generate_multimodal_report",
            "description": "Create a structured English report by combining information from multiple modalities (text, image, video, audio).",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Text describing the topic or content to include in the report. e.g. Summarize the security issues from this CCTV footage and transcript."
                    }
                },
                "required": ["query"],
                "additionalProperties": False
            }
        }
    }
]

completion = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {
            "role": "user", 
            #"content": "Dự báo thời tiết hiện tại ở Đà Nẵng"
            "content": "Give me an image of a cat similar to this image: a long-haired white cat sitting on a chair."
            #"content": "Give me the description of this video: time-lapse of clouds moving over a mountain at sunset"
            #"content": "Give me the report of this video CCTV footage showing a person entering a building at 10pm and leaving at 10:15pm"
        }
    ],
    tools=tool,
    tool_choice="auto"
    #tool_choice={"type": "function", "function": {"name": "generate_description"}}
    #tool_choice={"type": "function", "function": {"name": "generate_multimodal_report"}}
)
print(completion.choices[0].message.to_json())